# MCC month upload — Jan–Aug 2026 (с августом)

Готовая тетрадка для пересборки вкладки **«Аналитика MCC-кодов»**.

## Что делает
1. Грузит trx MCC помесячно (Jan–Aug 2026).
2. Собирает витрину `report_month × mcc` → CSV.
3. **Заливает в DRP** `sbx_da.tmp_shestopalov_acq_mcc_month` (+ GRANT `raisa_superset`).

## Как запустить
1. Kernel → **Restart & Run All** (или по ячейкам сверху вниз).
2. В DRP-ячейке ввести user/password.
3. Проверка в SQL Lab:

```sql
SELECT report_month, COUNT(*) AS n_mcc
FROM sbx_da.tmp_shestopalov_acq_mcc_month
GROUP BY 1
ORDER BY 1;
-- ожидаем 2026-01 … 2026-08
```

4. Superset: Dataset B → Sync columns → hard refresh вкладки MCC.

Период: `2026-01-01` ≤ d_trx < `2026-09-01` (август включён).  
`run_drp_upload_mcc_month = True` уже включён в config.

База логики: `mcc_trx_vs_merchant.ipynb` + `sources/sql/vd_acq_mcc_month.sql`.


# mcc_trx_vs_merchant — сравнение MCC + витрина MCC для дашборда

Сравниваем два источника MCC на зерне **`n_agr × report_month`** (Jan–Aug 2026):

| Источник | Таблица | Смысл |
|----------|--------|--------|
| **Merchant** | `ods_alpha.scd1_merchants.n_mcc` | справочник точки/компании (как в `final_script_2`) |
| **Trx** | `ods_alpha.scd1_trx.n_mcc` | код с операции (+ терминал `c_nter`) |

## Зачем
- У мерчантов часто multiple MCC (~1600 кейсов в `final_df`).
- Нужно понять: **trx-слой даёт меньше multiple?** и насколько множества совпадают.
- `final_df` / основная DRP-витрина договоров **пересобирать не нужно**.

## Гипотеза (проверяется цифрами в summary)
- Для дашборд-таблицы MCC (обороты / % эквайринга) → **trx**.
- Merchant MCC → справочно / drill-down.

## Outputs (сравнение)
- `mcc_compare_agr_month_2026_01_2026_08.csv`
- `mcc_compare_summary.csv`
- `mcc_compare_by_month_2026_01_2026_08.csv`

## Outputs (витрина дашборда «Аналитика MCC-кодов»)
- CSV: `vd_acq_mcc_month_2026_01_2026_08.csv` (зерно `report_month × mcc`)
- Опционально Impala: `sandbox_ai.shestopalov_acq_mcc_month`
- Опционально DRP: `sbx_da.tmp_shestopalov_acq_mcc_month` (`run_drp_upload_mcc_month = True`)


In [ ]:

import getpass
import re
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 120)

period_start = '2026-01-01'
period_end_exclusive = '2026-09-01'  # exclusive: включает август
period_months = pd.date_range(
    period_start,
    pd.to_datetime(period_end_exclusive) - pd.Timedelta(days=1),
    freq='MS',
)

output_dir = Path('/home/jovyan/documents/Equaring/Data')
if not output_dir.exists():
    output_dir = Path.cwd()
output_dir.mkdir(parents=True, exist_ok=True)

out_detail_csv = output_dir / 'mcc_compare_agr_month_2026_01_2026_08.csv'
out_summary_csv = output_dir / 'mcc_compare_summary.csv'
out_mcc_month_csv = output_dir / 'vd_acq_mcc_month_2026_01_2026_08.csv'

# Витрина MCC для дашборда (секция 6)
# False = только CSV локально; True = ещё CTAS / DRP
run_impala_ctas_mcc_month = False  # → sandbox_ai.shestopalov_acq_mcc_month
run_drp_upload_mcc_month = True    # → sbx_da.tmp_shestopalov_acq_mcc_month (+ GRANT)

impala_mcc_month_table = 'sandbox_ai.shestopalov_acq_mcc_month'
drp_mcc_schema = 'sbx_da'
drp_mcc_table = 'tmp_shestopalov_acq_mcc_month'
drp_superset_grant_role = 'raisa_superset'

print('period_months =', [m.strftime('%Y-%m') for m in period_months])
print('output_dir =', output_dir)
print('out_mcc_month_csv =', out_mcc_month_csv)
print('run_impala_ctas_mcc_month =', run_impala_ctas_mcc_month)
print('run_drp_upload_mcc_month =', run_drp_upload_mcc_month)


In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'},
)
imp._init_connection()
print('Impala connected')


## 1. Merchant MCC (`scd1_merchants`)

SA-договоры → все merchants компании (`n_cmp = n_cmp_client`) → distinct MCC на `n_agr`.

Merchant MCC **не зависит от месяца** (справочник), но для сравнения размножаем на каждый `report_month` периода (зерно agr×month).


In [ ]:
sql_merchant = """
with sa as (
  select
    cast(a.abs_agr_id as string) as agr_id,
    cast(a.n_agr as string) as n_agr,
    cast(a.n_cmp_client as string) as n_cmp_client
  from ods_alpha.scd1_agreements a
  where upper(trim(cast(a.acq_class as string))) = 'SA'
    and a.abs_agr_id is not null
)
select
  s.n_agr,
  s.agr_id,
  cast(m.n_mcc as string) as mcc
from sa s
left join ods_alpha.scd1_merchants m
  on cast(m.n_cmp as string) = s.n_cmp_client
where m.n_mcc is not null
  and trim(cast(m.n_mcc as string)) <> ''
"""

print('Loading merchant MCC map...')
with imp:
    merchant_raw = imp.fetch(sql_merchant)

if merchant_raw is None or len(merchant_raw) == 0:
    merchant_raw = pd.DataFrame(columns=['n_agr', 'agr_id', 'mcc'])

merchant_raw['n_agr'] = merchant_raw['n_agr'].astype(str)
merchant_raw['mcc'] = merchant_raw['mcc'].astype(str).str.strip()


def _join_unique(series):
    vals = sorted({
        str(v).strip()
        for v in series
        if v is not None and str(v).strip() not in ('', 'None', 'nan', 'NaN')
    })
    return ','.join(vals) if vals else None


merchant_by_agr = (
    merchant_raw.groupby('n_agr', as_index=False)
    .agg(
        agr_id=('agr_id', 'first'),
        mcc_merchant_list=('mcc', _join_unique),
        mcc_merchant_cnt=('mcc', lambda s: len({
            str(v).strip()
            for v in s
            if v is not None and str(v).strip() not in ('', 'None', 'nan', 'NaN')
        })),
    )
)

month_labels = [m.strftime('%Y-%m') for m in period_months]
merchant_month = (
    merchant_by_agr.assign(_k=1)
    .merge(pd.DataFrame({'report_month': month_labels, '_k': 1}), on='_k')
    .drop(columns='_k')
)

print('merchant agr with any MCC:', len(merchant_by_agr))
print('merchant agr with multi MCC:', int((merchant_by_agr['mcc_merchant_cnt'] > 1).sum()))
print('merchant_month rows:', len(merchant_month))
display(merchant_by_agr.sort_values('mcc_merchant_cnt', ascending=False).head(10))


## 2. Trx MCC (`scd1_trx.n_mcc`) — помесячно

Периметр как в `final_script_2` section 05 / `vd_acq_mcc_month.sql`:
SA + S01, RSHB acquirer, `c_nter` not null, не reject.

**Важно:** один запрос на весь Jan–Aug часто висит / жрёт 70–100+ GB.
Здесь цикл **по месяцу**: в SQL сразу агрегат `n_agr × mcc` (без выгрузки каждой trx-строки).

Если старый тяжёлый запрос ещё RUNNING в Impala — **Cancel** его в UI, затем запускайте эту ячейку.


In [ ]:
def _sql_trx_month(month_start: str, month_end_exclusive: str) -> str:
    """One calendar month; SQL returns agr×mcc aggregates (light)."""
    return f"""
with fiid_rshb as (
  select distinct cast(fa.c_fiid as string) as c_fiid
  from ods_alpha.scd1_base24_fiids fa
  where coalesce(cast(fa.c_fiid_grp as string), 'UNKNOWN') = 'RSHB'
),
sa_agr as (
  select distinct cast(a.n_agr as string) as n_agr
  from ods_alpha.scd1_agreements a
  where upper(trim(cast(a.acq_class as string))) = 'SA'
    and a.abs_agr_id is not null
),
trx_base_raw as (
  select
    cast(t.n_trx as string) as n_trx,
    cast(t.n_mcc as string) as mcc,
    cast(t.n_amt_src as double) as n_amt_src
  from ods_alpha.scd1_trx t
  join fiid_rshb fr
    on fr.c_fiid = cast(t.c_fiid_acq as string)
  where cast(t.d_trx_orig as timestamp) >= cast('{month_start}' as timestamp)
    and cast(t.d_trx_orig as timestamp) < cast('{month_end_exclusive}' as timestamp)
    and t.c_nter is not null
    and coalesce(t.ods_deleted_flg, '0') <> '1'
    and t.c_trx_class = 'SA'
    and t.c_trx_type = 'S01'
    and coalesce(t.cf_trx_stat, '') <> 'R'
    and t.n_mcc is not null
),
trx_base as (
  select n_trx, max(mcc) as mcc, max(n_amt_src) as n_amt_src
  from trx_base_raw
  group by n_trx
),
ta as (
  select
    cast(a.n_trx as string) as n_trx,
    cast(a.n_agr as string) as n_agr,
    max(coalesce(cast(a.n_amt_tax as double), 0.0)) as n_amt_tax
  from ods_alpha.scd1_trx_acq a
  join trx_base tb on tb.n_trx = cast(a.n_trx as string)
  join sa_agr ss on ss.n_agr = cast(a.n_agr as string)
  group by cast(a.n_trx as string), cast(a.n_agr as string)
)
select
  ta.n_agr,
  tb.mcc,
  count(distinct tb.n_trx) as trx_cnt,
  sum(tb.n_amt_src) as trx_sum,
  sum(ta.n_amt_tax) as commission_from_ops
from trx_base tb
join ta on ta.n_trx = tb.n_trx
group by ta.n_agr, tb.mcc
"""


print('Loading trx MCC month-by-month (cancel any hung full-period query first)...')
trx_parts = []
for m in period_months:
    m_start = m.strftime('%Y-%m-%d')
    m_end_excl = (m + pd.offsets.MonthBegin(1)).strftime('%Y-%m-%d')
    label = m.strftime('%Y-%m')
    print(f'  {label}: fetch...', flush=True)
    with imp:
        try:
            imp.execute('set MEM_LIMIT=16g')
        except Exception:
            pass
        part = imp.fetch(_sql_trx_month(m_start, m_end_excl))
    if part is None or len(part) == 0:
        print(f'  {label}: empty')
        continue
    part = part.copy()
    part['report_month'] = label
    print(f'  {label}: agr×mcc rows={len(part):,}')
    trx_parts.append(part)

if trx_parts:
    trx_mcc_month = pd.concat(trx_parts, ignore_index=True)
else:
    trx_mcc_month = pd.DataFrame(
        columns=['n_agr', 'mcc', 'trx_cnt', 'trx_sum', 'commission_from_ops', 'report_month']
    )

trx_mcc_month['n_agr'] = trx_mcc_month['n_agr'].astype(str)
trx_mcc_month['report_month'] = trx_mcc_month['report_month'].astype(str).str[:7]
trx_mcc_month['mcc'] = trx_mcc_month['mcc'].astype(str).str.strip()
for c in ['trx_cnt', 'trx_sum', 'commission_from_ops']:
    trx_mcc_month[c] = pd.to_numeric(trx_mcc_month[c], errors='coerce').fillna(0)

# primary MCC = max trx_sum per agr×month
primary_trx = (
    trx_mcc_month.sort_values(
        ['n_agr', 'report_month', 'trx_sum', 'mcc'],
        ascending=[True, True, False, True],
    )
    .groupby(['n_agr', 'report_month'], as_index=False)
    .first()
    .rename(columns={'mcc': 'mcc_trx_primary'})
    [['n_agr', 'report_month', 'mcc_trx_primary']]
)

trx_by_agr_month = (
    trx_mcc_month.groupby(['n_agr', 'report_month'], as_index=False)
    .agg(
        mcc_trx_list=('mcc', _join_unique),
        mcc_trx_cnt=('mcc', 'nunique'),
        trx_cnt=('trx_cnt', 'sum'),
        trx_sum=('trx_sum', 'sum'),
        commission_from_ops=('commission_from_ops', 'sum'),
    )
)
trx_by_agr_month = trx_by_agr_month.merge(primary_trx, on=['n_agr', 'report_month'], how='left')

print('trx agr×month rows:', len(trx_by_agr_month))
print('trx agr×month with multi MCC:', int((trx_by_agr_month['mcc_trx_cnt'] > 1).sum()))
display(trx_by_agr_month.sort_values('mcc_trx_cnt', ascending=False).head(10))


## 3. Compare merchant vs trx

Join на `n_agr × report_month`, флаги multiple, overlap множеств, primary MCC.


In [ ]:
def _mcc_set(s):
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return set()
    return {p.strip() for p in str(s).split(',') if p.strip() and p.strip() not in ('None', 'nan', 'NaN')}


cmp = merchant_month.merge(
    trx_by_agr_month,
    on=['n_agr', 'report_month'],
    how='outer',
    indicator=True,
)

cmp['mcc_merchant_cnt'] = pd.to_numeric(cmp['mcc_merchant_cnt'], errors='coerce').fillna(0).astype(int)
cmp['mcc_trx_cnt'] = pd.to_numeric(cmp['mcc_trx_cnt'], errors='coerce').fillna(0).astype(int)
cmp['trx_cnt'] = pd.to_numeric(cmp.get('trx_cnt'), errors='coerce').fillna(0)
cmp['trx_sum'] = pd.to_numeric(cmp.get('trx_sum'), errors='coerce').fillna(0)
cmp['commission_from_ops'] = pd.to_numeric(cmp.get('commission_from_ops'), errors='coerce').fillna(0)

cmp['merchant_multi'] = (cmp['mcc_merchant_cnt'] > 1).astype(int)
cmp['trx_multi'] = (cmp['mcc_trx_cnt'] > 1).astype(int)
cmp['has_merchant_mcc'] = (cmp['mcc_merchant_cnt'] > 0).astype(int)
cmp['has_trx_mcc'] = (cmp['mcc_trx_cnt'] > 0).astype(int)

sets_m = cmp['mcc_merchant_list'].map(_mcc_set)
sets_t = cmp['mcc_trx_list'].map(_mcc_set)
cmp['exact_set_match'] = [
    int(a == b and len(a) > 0) for a, b in zip(sets_m, sets_t)
]
cmp['trx_subset_of_merchant'] = [
    int(len(t) > 0 and t.issubset(m)) for m, t in zip(sets_m, sets_t)
]
cmp['merchant_subset_of_trx'] = [
    int(len(m) > 0 and m.issubset(t)) for m, t in zip(sets_m, sets_t)
]
cmp['intersection_cnt'] = [len(m & t) for m, t in zip(sets_m, sets_t)]

cmp['mcc_merchant_primary'] = cmp['mcc_merchant_list'].map(
    lambda s: str(s).split(',')[0] if pd.notna(s) and str(s).strip() else None
)
cmp['primary_match'] = (
    cmp['mcc_merchant_primary'].astype(str).str.strip()
    == cmp['mcc_trx_primary'].astype(str).str.strip()
).astype(int)
cmp.loc[cmp['mcc_trx_primary'].isna() | cmp['mcc_merchant_primary'].isna(), 'primary_match'] = 0

cmp['join_side'] = cmp['_merge'].astype(str)
cmp = cmp.drop(columns=['_merge'])

cols = [
    'report_month', 'n_agr', 'agr_id',
    'mcc_merchant_list', 'mcc_merchant_cnt', 'mcc_merchant_primary', 'merchant_multi',
    'mcc_trx_list', 'mcc_trx_cnt', 'mcc_trx_primary', 'trx_multi',
    'trx_cnt', 'trx_sum', 'commission_from_ops',
    'exact_set_match', 'trx_subset_of_merchant', 'merchant_subset_of_trx',
    'intersection_cnt', 'primary_match',
    'has_merchant_mcc', 'has_trx_mcc', 'join_side',
]
cmp = cmp[[c for c in cols if c in cmp.columns]].sort_values(['report_month', 'n_agr'])

print('compare rows (agr×month):', len(cmp))
display(cmp.head(15))
display(cmp.loc[cmp['merchant_multi'] == 1].head(10))


## 4. Summary — кто даёт меньше multiple / насколько совпадают

Рекомендация для дашборда MCC фиксируется по итогам прогона.


In [ ]:
with_both = cmp.loc[(cmp['has_merchant_mcc'] == 1) & (cmp['has_trx_mcc'] == 1)].copy()

agr_month_merchant_multi = int((cmp['merchant_multi'] == 1).sum())
agr_month_trx_multi = int((cmp['trx_multi'] == 1).sum())
delta_multi = agr_month_merchant_multi - agr_month_trx_multi

trx_universe = cmp.loc[cmp['has_trx_mcc'] == 1]
merchant_multi_among_trx = int((trx_universe['merchant_multi'] == 1).sum())
trx_multi_among_trx = int((trx_universe['trx_multi'] == 1).sum())

n_both = len(with_both)
share_exact = float(with_both['exact_set_match'].mean()) if n_both else None
share_trx_subset = float(with_both['trx_subset_of_merchant'].mean()) if n_both else None
share_primary = float(with_both['primary_match'].mean()) if n_both else None

only_merchant_multi = int(((cmp['merchant_multi'] == 1) & (cmp['trx_multi'] == 0)).sum())
only_trx_multi = int(((cmp['trx_multi'] == 1) & (cmp['merchant_multi'] == 0)).sum())
both_multi = int(((cmp['merchant_multi'] == 1) & (cmp['trx_multi'] == 1)).sum())

summary = pd.DataFrame([{
    'agr_month_rows': len(cmp),
    'agr_month_with_merchant_mcc': int(cmp['has_merchant_mcc'].sum()),
    'agr_month_with_trx_mcc': int(cmp['has_trx_mcc'].sum()),
    'agr_month_with_both': n_both,
    'agr_month_merchant_multi': agr_month_merchant_multi,
    'agr_month_trx_multi': agr_month_trx_multi,
    'delta_multi_merchant_minus_trx': delta_multi,
    'merchant_multi_among_trx_active': merchant_multi_among_trx,
    'trx_multi_among_trx_active': trx_multi_among_trx,
    'only_merchant_multi': only_merchant_multi,
    'only_trx_multi': only_trx_multi,
    'both_sides_multi': both_multi,
    'share_exact_set_match': share_exact,
    'share_trx_subset_of_merchant': share_trx_subset,
    'share_primary_mcc_match': share_primary,
}])

print('=== MCC compare summary (agr × month) ===')
display(summary.T.rename(columns={0: 'value'}))

by_month = (
    cmp.groupby('report_month', as_index=False)
    .agg(
        rows=('n_agr', 'size'),
        merchant_multi=('merchant_multi', 'sum'),
        trx_multi=('trx_multi', 'sum'),
        with_trx=('has_trx_mcc', 'sum'),
        exact_match=('exact_set_match', 'sum'),
    )
)
by_month['delta_multi'] = by_month['merchant_multi'] - by_month['trx_multi']
print('=== by month ===')
display(by_month)

if agr_month_trx_multi < agr_month_merchant_multi:
    reco = (
        'RECOMMEND: для дашборд-таблицы MCC использовать trx-слой '
        '(меньше agr×month с multiple MCC). Merchant MCC — справочно.'
    )
elif agr_month_trx_multi > agr_month_merchant_multi:
    reco = (
        'UNEXPECTED: trx multiple > merchant multiple — перепроверьте периметр trx / join на n_agr.'
    )
else:
    reco = (
        'MULTIPLE counts equal — смотрите overlap и primary_match; '
        'для объёмов всё равно предпочтителен trx.'
    )

print(reco)
print(
    f"multi merchant={agr_month_merchant_multi:,} | "
    f"multi trx={agr_month_trx_multi:,} | "
    f"delta={delta_multi:,} | "
    f"exact_match={share_exact} | "
    f"trx_subset_merchant={share_trx_subset}"
)

cmp.to_csv(out_detail_csv, index=False, encoding='utf-8-sig')
summary.to_csv(out_summary_csv, index=False, encoding='utf-8-sig')
by_month_path = output_dir / 'mcc_compare_by_month_2026_01_2026_08.csv'
by_month.to_csv(by_month_path, index=False, encoding='utf-8-sig')
print('saved:', out_detail_csv)
print('saved:', out_summary_csv)
print('saved:', by_month_path)


## 5. Как читать результат сравнения

1. Смотрите `delta_multi_merchant_minus_trx` в summary: если **> 0**, у merchant больше multiple — как ожидалось.
2. Среди активных по trx (`with_trx`) сравните `merchant_multi_among_trx_active` vs `trx_multi_among_trx_active`.
3. Высокий `share_trx_subset_of_merchant` значит trx-коды обычно уже есть в справочнике мерчанта (справочник шире).
4. Витрина для дашборда «Аналитика MCC» — **секция 6 ниже** (зерно `month × mcc`).
5. `final_df` с merchant MCC **не обязательно** пересобирать.


## 6. Витрина дашборда: `vd_acq_mcc_month` (месяц × MCC)

Логика как в `sources/sql/vd_acq_mcc_month.sql`, но **без отдельного SQL Lab**.

Зерно: **`report_month × mcc`**  
Колонки под таблицу дашборда:
- `mcc` — MCC-код
- `acq_pct` — средний % эквайринга (commission / trx_sum × 100)
- `share_trx_sum_pct` — доля в общем объёме операций за месяц

**Как считается:** из уже загруженного `trx_mcc_month` (секция 2) агрегируем по месяцу и MCC в pandas.  
Повторный тяжёлый запрос в Impala **не нужен**, если секция 2 уже отработала.

Если `trx_mcc_month` нет в памяти — ячейка сама догрузит помесячно (только агрегат по MCC).

Дальше:
1. Проверьте превью / CSV.
2. При необходимости: `run_impala_ctas_mcc_month = True` (секция config) → таблица в Impala.
3. Для Superset на DRP: `run_drp_upload_mcc_month = True` → `sbx_da.tmp_shestopalov_acq_mcc_month`.


In [ ]:

def _sql_mcc_only_month(month_start: str, month_end_exclusive: str) -> str:
    """One month → grain mcc (for dashboard). Fallback if trx_mcc_month missing."""
    return f"""
with fiid_rshb as (
  select distinct cast(fa.c_fiid as string) as c_fiid
  from ods_alpha.scd1_base24_fiids fa
  where coalesce(cast(fa.c_fiid_grp as string), 'UNKNOWN') = 'RSHB'
),
sa_agr as (
  select distinct cast(a.n_agr as string) as n_agr
  from ods_alpha.scd1_agreements a
  where upper(trim(cast(a.acq_class as string))) = 'SA'
    and a.abs_agr_id is not null
),
trx_base_raw as (
  select
    cast(t.n_trx as string) as n_trx,
    cast(t.n_mcc as string) as mcc,
    cast(t.n_amt_src as double) as n_amt_src
  from ods_alpha.scd1_trx t
  join fiid_rshb fr
    on fr.c_fiid = cast(t.c_fiid_acq as string)
  where cast(t.d_trx_orig as timestamp) >= cast('{month_start}' as timestamp)
    and cast(t.d_trx_orig as timestamp) < cast('{month_end_exclusive}' as timestamp)
    and t.c_nter is not null
    and coalesce(t.ods_deleted_flg, '0') <> '1'
    and t.c_trx_class = 'SA'
    and t.c_trx_type = 'S01'
    and coalesce(t.cf_trx_stat, '') <> 'R'
    and t.n_mcc is not null
),
trx_base as (
  select n_trx, max(mcc) as mcc, max(n_amt_src) as n_amt_src
  from trx_base_raw
  group by n_trx
),
ta as (
  select
    cast(a.n_trx as string) as n_trx,
    cast(a.n_agr as string) as n_agr,
    max(coalesce(cast(a.n_amt_tax as double), 0.0)) as n_amt_tax
  from ods_alpha.scd1_trx_acq a
  join trx_base tb on tb.n_trx = cast(a.n_trx as string)
  join sa_agr ss on ss.n_agr = cast(a.n_agr as string)
  group by cast(a.n_trx as string), cast(a.n_agr as string)
)
select
  tb.mcc,
  count(distinct tb.n_trx) as trx_cnt,
  sum(tb.n_amt_src) as trx_sum,
  sum(ta.n_amt_tax) as commission_from_ops
from trx_base tb
join ta on ta.n_trx = tb.n_trx
group by tb.mcc
"""


def _build_mcc_month_from_agr_mcc(trx_agr_mcc: pd.DataFrame) -> pd.DataFrame:
    """rollup agr×mcc×month → month×mcc (+ metrics like vd_acq_mcc_month.sql)."""
    g = (
        trx_agr_mcc.groupby(['report_month', 'mcc'], as_index=False)
        .agg(
            trx_cnt=('trx_cnt', 'sum'),
            trx_sum=('trx_sum', 'sum'),
            commission_from_ops=('commission_from_ops', 'sum'),
        )
    )
    g['snapshot_month_start'] = g['report_month'].astype(str).str[:7] + '-01'
    month_trx = g.groupby('report_month')['trx_sum'].transform('sum')
    month_cnt = g.groupby('report_month')['trx_cnt'].transform('sum')
    g['acq_pct'] = np.where(
        g['trx_sum'] == 0,
        np.nan,
        g['commission_from_ops'] / g['trx_sum'] * 100.0,
    )
    g['share_trx_sum_pct'] = np.where(month_trx == 0, np.nan, g['trx_sum'] / month_trx * 100.0)
    g['share_trx_cnt_pct'] = np.where(month_cnt == 0, np.nan, g['trx_cnt'] / month_cnt * 100.0)
    cols = [
        'report_month', 'snapshot_month_start', 'mcc',
        'trx_cnt', 'trx_sum', 'commission_from_ops',
        'acq_pct', 'share_trx_sum_pct', 'share_trx_cnt_pct',
    ]
    return g[cols].sort_values(['report_month', 'trx_sum'], ascending=[True, False]).reset_index(drop=True)


# Prefer reuse of section-2 result (no second heavy Impala pass)
if 'trx_mcc_month' in globals() and trx_mcc_month is not None and len(trx_mcc_month) > 0:
    print('Building vd_acq_mcc_month from in-memory trx_mcc_month (no Impala refetch)...')
    mcc_month_df = _build_mcc_month_from_agr_mcc(trx_mcc_month)
else:
    print('trx_mcc_month missing — fetching month×mcc from Impala month-by-month...')
    parts = []
    for m in period_months:
        m_start = m.strftime('%Y-%m-%d')
        m_end_excl = (m + pd.offsets.MonthBegin(1)).strftime('%Y-%m-%d')
        label = m.strftime('%Y-%m')
        print(f'  {label}: fetch...', flush=True)
        with imp:
            try:
                imp.execute('set MEM_LIMIT=16g')
            except Exception:
                pass
            part = imp.fetch(_sql_mcc_only_month(m_start, m_end_excl))
        if part is None or len(part) == 0:
            print(f'  {label}: empty')
            continue
        part = part.copy()
        part['report_month'] = label
        print(f'  {label}: mcc rows={len(part):,}')
        parts.append(part)
    if not parts:
        mcc_month_df = pd.DataFrame(
            columns=[
                'report_month', 'snapshot_month_start', 'mcc',
                'trx_cnt', 'trx_sum', 'commission_from_ops',
                'acq_pct', 'share_trx_sum_pct', 'share_trx_cnt_pct',
            ]
        )
    else:
        raw = pd.concat(parts, ignore_index=True)
        raw['mcc'] = raw['mcc'].astype(str).str.strip()
        for c in ['trx_cnt', 'trx_sum', 'commission_from_ops']:
            raw[c] = pd.to_numeric(raw[c], errors='coerce').fillna(0)
        mcc_month_df = _build_mcc_month_from_agr_mcc(raw)

print('mcc_month_df rows =', len(mcc_month_df))
print('months =', sorted(mcc_month_df['report_month'].dropna().unique().tolist()) if len(mcc_month_df) else [])
display(mcc_month_df.head(20))

# sanity: shares per month ~ 100%
if len(mcc_month_df):
    share_check = (
        mcc_month_df.groupby('report_month', as_index=False)
        .agg(share_sum=('share_trx_sum_pct', 'sum'), trx_sum=('trx_sum', 'sum'), mcc_n=('mcc', 'nunique'))
    )
    display(share_check)


In [ ]:

mcc_month_df.to_csv(out_mcc_month_csv, index=False, encoding='utf-8-sig')
print('saved:', out_mcc_month_csv)
print('columns:', list(mcc_month_df.columns))


In [ ]:

# Optional: materialize in Impala (CTAS), same logic as vd_acq_mcc_month.sql
if not run_impala_ctas_mcc_month:
    print('SKIP Impala CTAS (run_impala_ctas_mcc_month=False)')
else:
    if mcc_month_df is None or len(mcc_month_df) == 0:
        raise RuntimeError('mcc_month_df пустой — сначала соберите секцию 6.')

    # Build SELECT body without ORDER BY (Impala CTAS)
    sql_select = f"""
with params as (
  select
    cast('{period_start}' as timestamp) as period_start,
    cast('{period_end_exclusive}' as timestamp) as period_end_exclusive
),
fiid_rshb as (
  select distinct cast(fa.c_fiid as string) as c_fiid
  from ods_alpha.scd1_base24_fiids fa
  where coalesce(cast(fa.c_fiid_grp as string), 'UNKNOWN') = 'RSHB'
),
sa_agr as (
  select distinct cast(a.n_agr as string) as n_agr
  from ods_alpha.scd1_agreements a
  where upper(trim(cast(a.acq_class as string))) = 'SA'
    and a.abs_agr_id is not null
),
trx_base_raw as (
  select
    cast(t.n_trx as string) as n_trx,
    cast(t.n_mcc as string) as mcc,
    cast(t.n_amt_src as double) as n_amt_src,
    trunc(cast(t.d_trx_orig as timestamp), 'MM') as report_month_ts
  from ods_alpha.scd1_trx t
  join fiid_rshb fr
    on fr.c_fiid = cast(t.c_fiid_acq as string)
  cross join params p
  where cast(t.d_trx_orig as timestamp) >= p.period_start
    and cast(t.d_trx_orig as timestamp) < p.period_end_exclusive
    and t.c_nter is not null
    and coalesce(t.ods_deleted_flg, '0') <> '1'
    and t.c_trx_class = 'SA'
    and t.c_trx_type = 'S01'
    and coalesce(t.cf_trx_stat, '') <> 'R'
    and t.n_mcc is not null
),
trx_base as (
  select n_trx, max(mcc) as mcc, max(n_amt_src) as n_amt_src, max(report_month_ts) as report_month_ts
  from trx_base_raw
  group by n_trx
),
ta as (
  select
    cast(a.n_trx as string) as n_trx,
    cast(a.n_agr as string) as n_agr,
    max(coalesce(cast(a.n_amt_tax as double), 0.0)) as n_amt_tax
  from ods_alpha.scd1_trx_acq a
  join trx_base tb on tb.n_trx = cast(a.n_trx as string)
  join sa_agr ss on ss.n_agr = cast(a.n_agr as string)
  group by cast(a.n_trx as string), cast(a.n_agr as string)
),
trx_mcc as (
  select
    substr(cast(tb.report_month_ts as string), 1, 7) as report_month,
    cast(tb.report_month_ts as string) as snapshot_month_start,
    tb.mcc,
    tb.n_trx,
    tb.n_amt_src,
    ta.n_amt_tax
  from trx_base tb
  join ta on ta.n_trx = tb.n_trx
),
agg as (
  select
    report_month,
    snapshot_month_start,
    mcc,
    count(distinct n_trx) as trx_cnt,
    sum(n_amt_src) as trx_sum,
    sum(n_amt_tax) as commission_from_ops
  from trx_mcc
  group by report_month, snapshot_month_start, mcc
)
select
  report_month,
  snapshot_month_start,
  mcc,
  trx_cnt,
  trx_sum,
  commission_from_ops,
  case when trx_sum is null or trx_sum = 0 then null
       else commission_from_ops / trx_sum * 100.0 end as acq_pct,
  case when sum(trx_sum) over (partition by report_month) = 0 then null
       else trx_sum / sum(trx_sum) over (partition by report_month) * 100.0 end as share_trx_sum_pct,
  case when sum(trx_cnt) over (partition by report_month) = 0 then null
       else trx_cnt * 1.0 / sum(trx_cnt) over (partition by report_month) * 100.0 end as share_trx_cnt_pct
from agg
"""

    print('Impala CTAS →', impala_mcc_month_table)
    print('WARNING: full-period CTAS may be heavy; prefer pandas+DRP path if it hangs.')
    with imp:
        try:
            imp.execute('set MEM_LIMIT=16g')
        except Exception:
            pass
        imp.execute(f'DROP TABLE IF EXISTS {impala_mcc_month_table}')
        imp.execute(
            f'CREATE TABLE {impala_mcc_month_table} STORED AS PARQUET AS\n{sql_select}'
        )
        try:
            imp.execute(f'invalidate metadata {impala_mcc_month_table}')
        except Exception:
            pass
        chk = imp.fetch(f'select count(*) as row_cnt from {impala_mcc_month_table}')
    print('OK Impala CTAS rows =', int(chk.iloc[0, 0]) if chk is not None and len(chk) else None)


In [ ]:

# Optional: upload MCC month mart to DRP for Superset
if not run_drp_upload_mcc_month:
    print('SKIP DRP upload MCC (run_drp_upload_mcc_month=False)')
    print('Когда будете готовы: в config поставьте True и перезапустите эту ячейку.')
else:
    if 'mcc_month_df' not in globals() or mcc_month_df is None or len(mcc_month_df) == 0:
        raise RuntimeError('Сначала соберите mcc_month_df (секция 6).')

    target_fq = f'{drp_mcc_schema}.{drp_mcc_table}'
    print('Preparing DRP upload →', target_fq)
    print('source rows =', len(mcc_month_df))

    drp_user = input('DRP user: ').strip()
    drp_password = getpass.getpass('DRP password: ')

    drp_conn = connect(
        to='DRP',
        user_params={
            'user_name': drp_user,
            'password': drp_password,
        },
    )

    upload_df = mcc_month_df.copy()
    upload_df.columns = [str(c).strip().lower() for c in upload_df.columns]
    for c in upload_df.columns:
        upload_df[c] = upload_df[c].map(lambda x: None if pd.isna(x) else str(x)).astype(object)

    col_defs = [f'"{str(c).replace(chr(34), chr(34)+chr(34))}" TEXT' for c in upload_df.columns]
    create_sql = f'CREATE TABLE {target_fq} (\n  ' + ',\n  '.join(col_defs) + '\n)'

    with drp_conn:
        drp_conn.execute(f'DROP TABLE IF EXISTS {target_fq}')
        drp_conn.execute(create_sql)
        drp_conn.write(table=target_fq, df=upload_df, mode='append')

        cnt_df = drp_conn.fetch(f'select count(*) as row_cnt from {target_fq}')
        try:
            drp_conn.execute(f'GRANT USAGE ON SCHEMA {drp_mcc_schema} TO {drp_superset_grant_role}')
            drp_conn.execute(f'GRANT SELECT ON TABLE {target_fq} TO {drp_superset_grant_role}')
            print(f'OK: GRANT SELECT ON {target_fq} TO {drp_superset_grant_role}')
        except Exception as grant_exc:
            print('WARNING: GRANT failed:', type(grant_exc).__name__, str(grant_exc)[:400])
            print(f'  GRANT SELECT ON TABLE {target_fq} TO {drp_superset_grant_role};')

    row_cnt = int(pd.to_numeric(cnt_df.iloc[0, 0], errors='coerce')) if cnt_df is not None and len(cnt_df) else 0
    print('OK DRP rows =', row_cnt)
    print('Superset: новый dataset на', target_fq)
    print('Чарт: dimension mcc; metrics acq_pct, share_trx_sum_pct; filter report_month')


## 7. Что запускать для дашборда (кратко)

| Цель | Ячейки | Флаги в config |
|------|--------|----------------|
| Только сравнение merchant vs trx | 1 → 5 | оба False |
| CSV витрины MCC (уже есть `trx_mcc_month`) | секция 6 (build + save) | оба False |
| + таблица в Impala | CTAS-ячейка | `run_impala_ctas_mcc_month = True` |
| + витрина в DRP для Superset | DRP-ячейка | `run_drp_upload_mcc_month = True` |

Рекомендуемый путь для дашборда на DRP:
1. Прогнать секции 1–2 (чтобы был `trx_mcc_month`) **или** только секцию 6 (сама догрузит).
2. Собрать / сохранить `mcc_month_df`.
3. В config уже `run_drp_upload_mcc_month = True` — после сборки витрины запусти DRP-ячейку (логин/пароль).
4. В Superset — dataset `sbx_da.tmp_shestopalov_acq_mcc_month`.

`final_script_2` менять не нужно.
